In [3]:
!pip install google-generativeai pandas --quiet


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import google.generativeai as genai
import pandas as pd
import json
import time
from getpass import getpass
from google.api_core.exceptions import ResourceExhausted

api_key = getpass("Enter your Gemini API key: ")
genai.configure(api_key=api_key)

MODEL = "gemini-3.6-flash"

def ask(prompt, system=None, max_tokens=500):
    """Send a single prompt to the LLM and return the text response.
    - Always leaves extra headroom for the model's internal 'thinking' tokens,
      which count against max_output_tokens on this model and can otherwise
      eat the whole budget before any visible text is produced.
    - Automatically waits and retries if the free-tier rate limit (429) is hit,
      instead of crashing the whole notebook.
    """
    model = genai.GenerativeModel(MODEL, system_instruction=system)
    safe_max_tokens = max(max_tokens, 300)  # floor so short prompts still get room to think

    for attempt in range(4):
        try:
            response = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": safe_max_tokens},
            )
            try:
                return response.text
            except ValueError:
                try:
                    reason = response.candidates[0].finish_reason
                except Exception:
                    reason = "unknown"
                return f"[NO TEXT RETURNED - finish_reason: {reason}]"
        except ResourceExhausted:
            wait = 20 * (attempt + 1)
            print(f"Rate limit hit, waiting {wait}s before retrying (attempt {attempt + 1}/4)...")
            time.sleep(wait)

    return "[RATE LIMITED - free tier quota exceeded after retries. Wait a bit longer and re-run this cell.]"


In [5]:
sample = "Electric vehicle adoption is accelerating faster than analysts predicted."
model = genai.GenerativeModel(MODEL)
count = model.count_tokens(sample)
print(f"'{sample}'")
print(f"-> {count.total_tokens} tokens")


'Electric vehicle adoption is accelerating faster than analysts predicted.'
-> 10 tokens


In [6]:
articles_df = pd.read_csv("summarization_articles.csv")
print(f"{len(articles_df)} articles available. Using the first one:\n")

row = articles_df.iloc[0]
article = row["article"]
gold_summary = row["gold_summary"]

print("HEADLINE:", row["headlines"])
print()
print("ARTICLE:\n", article)
print()
print("DATASET'S OWN (human-written) GOLD SUMMARY:\n", gold_summary)


15 articles available. Using the first one:

HEADLINE: Chopper flying critically low led to 2015 Bombay High crash

ARTICLE:
 Two years after a helicopter crash near the Bombay High offshore oil field killed two pilots, an inquiry by the Air Accident Investigation Bureau (AAIB) found that the chopper was flying at a critically low height, without the pilots realising it. The Pawan Hans helicopter was on a training sortie on the evening of November 4, 2015. Aviation regulator Director General of Civil Aviation (DGCA) had asked AAIB?to conduct an inquiry. The inquiry report, which was made public on Wednesday, said that the main pilot was flying low and that the other did not realise it. After taking off, the Rontappmeyer (RYTYR) helicopter may have faced difficulties as the moon was waning, the report said. The report mentioned that the instructor, Captain E Samuel, wasn?t disoriented and was aware of the low height of the helicopter. He alerted Captain TK Guha a little late, but even a

In [7]:
# --- Prompt A: bare-minimum instruction ---
prompt_a = f"Summarize this:\n\n{article}"
summary_a = ask(prompt_a)
print("PROMPT A OUTPUT:\n", summary_a)


Rate limit hit, waiting 20s before retrying (attempt 1/4)...
Rate limit hit, waiting 40s before retrying (attempt 2/4)...
Rate limit hit, waiting 60s before retrying (attempt 3/4)...
Rate limit hit, waiting 80s before retrying (attempt 4/4)...
PROMPT A OUTPUT:
 [RATE LIMITED - free tier quota exceeded after retries. Wait a bit longer and re-run this cell.]


In [8]:
# --- Prompt B: constrained length + focus ---
prompt_b = f"""Summarize the following news article in exactly 2 sentences.
Focus only on: (1) what happened, and (2) why it matters.

Article:
{article}"""
summary_b = ask(prompt_b)
print("PROMPT B OUTPUT:\n", summary_b)


Rate limit hit, waiting 20s before retrying (attempt 1/4)...
Rate limit hit, waiting 40s before retrying (attempt 2/4)...
Rate limit hit, waiting 60s before retrying (attempt 3/4)...
PROMPT B OUTPUT:
 An Air Accident Investigation Bureau inquiry revealed that a 2015 Pawan Hans helicopter crash near the


In [9]:
# --- Prompt C: role + audience + format ---
prompt_c = f"""You are a news editor writing a one-line summary for a mobile push
notification. In no more than 30 words, capture the single most newsworthy fact.
Output plain text, no preamble.

Article:
{article}"""
summary_c = ask(prompt_c)
print("PROMPT C OUTPUT:\n", summary_c)


Rate limit hit, waiting 20s before retrying (attempt 1/4)...
PROMPT C OUTPUT:
 An official inquiry into the fatal 2015 Pawan Hans helicopter crash


In [ ]:
def run_n_times(prompt, n=3):
    outputs = []
    for i in range(n):
        out = ask(prompt)
        outputs.append(out)
        print(f"--- Run {i+1} ({len(out.split())} words) ---")
        print(out)
        print()
    return outputs

print("### Prompt A (bare) x3 ###\n")
runs_a = run_n_times(prompt_a)


### Prompt A (bare) x3 ###

Rate limit hit, waiting 20s before retrying (attempt 1/4)...
Rate limit hit, waiting 40s before retrying (attempt 2/4)...


In [ ]:
print("### Prompt C (role + word cap) x3 ###\n")
runs_c = run_n_times(prompt_c)

word_counts_a = [len(o.split()) for o in runs_a]
word_counts_c = [len(o.split()) for o in runs_c]
print(f"Prompt A word counts across 3 runs: {word_counts_a}  (range: {max(word_counts_a)-min(word_counts_a)})")
print(f"Prompt C word counts across 3 runs: {word_counts_c}  (range: {max(word_counts_c)-min(word_counts_c)})")


### Prompt C (role + word cap) x3 ###

--- Run 1 (15 words) ---
An AAIB probe into the 2015 Pawan Hans helicopter crash near Bombay High found the

--- Run 2 (14 words) ---
An official inquiry revealed the fatal 2015 Pawan Hans helicopter crash off Bombay High

--- Run 3 (14 words) ---
An inquiry report reveals the 2015 Pawan Hans helicopter crash off Bombay High occurred

Prompt A word counts across 3 runs: [14, 9, 12]  (range: 5)
Prompt C word counts across 3 runs: [15, 14, 14]  (range: 1)


In [ ]:
tickets_df = pd.read_csv("classification_tickets.csv")
print(tickets_df.shape)
print(tickets_df["label"].value_counts())
print()

# Sample a mix across categories to keep API calls manageable
sample_df = tickets_df.groupby("label").sample(2, random_state=7).reset_index(drop=True)
tickets = sample_df["text"].tolist()
ground_truth = sample_df["label"].tolist()
categories = sorted(tickets_df["label"].unique().tolist())

for t, g in zip(tickets, ground_truth):
    print(f"[{g:10s}] {t}")


(50, 4)
label
login       10
refund      10
bug         10
billing     10
delivery    10
Name: count, dtype: int64

[billing   ] How do I pause my subscription?
[billing   ] Change my credit card details
[bug       ] The app freezes on the loading screen
[bug       ] I get an error when submitting a form
[delivery  ] Item was damaged in transit
[delivery  ] Delivery is late
[login     ] Authentication error when logging in
[login     ] Can't remember my login credentials
[refund    ] Refund the item I returned
[refund    ] Please cancel and refund my order


In [ ]:
# --- Prompt A: zero-shot, no definitions ---
def classify_a(ticket):
    prompt = f"""Classify this support ticket into one category: {categories}.
Ticket: "{ticket}"
Category:"""
    return ask(prompt, max_tokens=10).strip().lower()

preds_a = [classify_a(t) for t in tickets]
for t, p in zip(tickets, preds_a):
    print(f"{p:12s} <- {t}")


billing      <- How do I pause my subscription?
billing      <- Change my credit card details
bug          <- The app freezes on the loading screen
bug          <- I get an error when submitting a form
delivery     <- Item was damaged in transit
delivery     <- Delivery is late
login        <- Authentication error when logging in
login        <- Can't remember my login credentials
refund       <- Refund the item I returned
refund       <- Please cancel and refund my order


In [ ]:
# --- Prompt B: zero-shot with explicit category definitions ---
def classify_b(ticket):
    prompt = f"""Classify the support ticket below into exactly one of these categories:

- login: password resets, sign-in failures, account access, authentication issues
- refund: requests for money back, reimbursements, cancellations
- bug: app crashes, errors, broken features, unexpected behavior
- billing: subscription/payment method changes, invoice/charge questions
- delivery: shipping, package tracking, wrong or missing items

Respond with only the category name in lowercase, nothing else.

Ticket: "{ticket}"
Category:"""
    return ask(prompt, max_tokens=10).strip().lower()

preds_b = [classify_b(t) for t in tickets]
for t, p in zip(tickets, preds_b):
    print(f"{p:12s} <- {t}")


NameError: name 'tickets' is not defined

In [ ]:
# --- Prompt C: few-shot + structured JSON output ---
def classify_c(ticket):
    prompt = f"""Classify customer support tickets into one of: login, refund, bug, billing, delivery.

Examples:
Ticket: "Can't remember my login credentials"
{{"category": "login", "confidence": "high"}}

Ticket: "My package never arrived"
{{"category": "delivery", "confidence": "high"}}

Now classify this ticket. Respond with ONLY a JSON object like the examples above, no other text.

Ticket: "{ticket}"
"""
    return ask(prompt, max_tokens=40).strip()

raw_c = [classify_c(t) for t in tickets]
for t, raw in zip(tickets, raw_c):
    print(f"{raw:45s} <- {t}")


Rate limit hit, waiting 20s before retrying (attempt 1/4)...
Rate limit hit, waiting 40s before retrying (attempt 2/4)...
Rate limit hit, waiting 60s before retrying (attempt 3/4)...
Rate limit hit, waiting 80s before retrying (attempt 4/4)...
Rate limit hit, waiting 20s before retrying (attempt 1/4)...
Rate limit hit, waiting 40s before retrying (attempt 2/4)...
Rate limit hit, waiting 60s before retrying (attempt 3/4)...


In [ ]:
# Score each prompt against the real dataset labels
def score(preds, labels, is_json=False):
    correct = 0
    for p, label in zip(preds, labels):
        if is_json:
            try:
                cat = json.loads(p)["category"].lower()
            except (json.JSONDecodeError, KeyError, AttributeError):
                cat = None
        else:
            cat = p
        if cat == label:
            correct += 1
    return correct / len(labels)

acc_a = score(preds_a, ground_truth)
acc_b = score(preds_b, ground_truth)
acc_c = score(raw_c, ground_truth, is_json=True)

print(f"Prompt A accuracy (no definitions):    {acc_a:.0%}")
print(f"Prompt B accuracy (+ definitions):      {acc_b:.0%}")
print(f"Prompt C accuracy (+ few-shot, JSON):    {acc_c:.0%}")


NameError: name 'preds_a' is not defined

In [ ]:
tricky_ticket = "I was charged again after I already cancelled and returned the item -- not sure if this is a refund or billing problem."
# Ambiguous by design: it references both a charge (billing) and a cancellation/return (refund).

print("Prompt B:", classify_b(tricky_ticket))
print("Prompt C:", classify_c(tricky_ticket))


NameError: name 'classify_b' is not defined

In [ ]:
with open("extraction_resumes.json") as f:
    resumes = json.load(f)

for r in resumes:
    print("====", r["Category"], "====")
    print(r["Resume"][:300], "...\n")


NameError: name 'json' is not defined

In [ ]:
# --- Prompt A: naive extraction, no schema (tested on the first resume only) ---
prompt_a_extract = f"Extract the important information from this resume:\n\n{resumes[0]['Resume']}"
print(ask(prompt_a_extract))


NameError: name 'resumes' is not defined

In [ ]:
# --- Prompt B: explicit field list, still free text (tested on the first resume only) ---
prompt_b_extract = f"""Extract the following fields from the resume. If a field isn't mentioned, write "N/A".

- Education (degree, institution, year)
- Key skills (list)
- Most recent job title
- Years of experience (estimate if not explicit)
- Notable certifications or recognitions

Resume:
{resumes[0]['Resume']}"""
print(ask(prompt_b_extract))


NameError: name 'resumes' is not defined

In [ ]:
# --- Prompt C: strict JSON schema with types, run across ALL THREE resumes ---
def extract_c(resume_text):
    prompt = f"""Extract information from the resume into this exact JSON schema. Use null for
missing fields and empty arrays for missing lists. Output ONLY valid JSON, no markdown fences, no explanation.

{{
  "education": [{{"degree": string, "institution": string or null, "year": string or null}}],
  "skills": [string],
  "most_recent_title": string or null,
  "estimated_years_experience": number or null,
  "certifications": [string]
}}

Resume:
{resume_text}"""
    return ask(prompt, max_tokens=500)

results_c = [extract_c(r["Resume"]) for r in resumes]
for i, (r, res) in enumerate(zip(resumes, results_c)):
    print(f"--- Resume {i+1} ({r['Category']}) ---")
    print(res)
    print()


NameError: name 'resumes' is not defined

In [ ]:
# Score Prompt C by how many of the 3 resumes produce valid, parseable JSON
valid_count = 0
for i, r in enumerate(results_c):
    try:
        parsed = json.loads(r)
        valid_count += 1
        print(f"Resume {i+1}: valid JSON ->")
        print(json.dumps(parsed, indent=2))
    except json.JSONDecodeError as e:
        print(f"Resume {i+1}: FAILED to parse -- {e}")
    print()

print(f"\nJSON validity rate: {valid_count}/{len(resumes)} ({valid_count/len(resumes):.0%})")


NameError: name 'results_c' is not defined